# BFT65B - Part 3 (step 2): the same pipeline in PySpark, compared with Polars
Runs on Google Colab (free). Steps: **Runtime > Run all**, and when the upload box appears choose the two files
`panel_spark.parquet` (from `data_clean/`, made by `06_export_for_spark.py`) and `polars_check.csv` (from `reports/part3/`).

What this notebook does: (1) rebuilds the feature pipeline of `05_polars_vs_pandas.py` with Spark DataFrames and window functions,
(2) checks that the results equal the Polars results, (3) prints and explains the Spark execution plans,
(4) times Spark and Polars on the **same** Colab machine.

In [1]:
%pip install -q pyspark polars

In [2]:
# Spark 4 needs Java 17 or newer: check, and install it if necessary (takes about one minute)
import re, subprocess
def java_major():
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        return int(re.search(r'version "(\d+)', out).group(1))
    except Exception:
        return 0
if java_major() < 17:
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless", shell=True)
print("Java major version:", java_major())

Java major version: 21


In [3]:
# Upload the two files (skipped if they are already in the working folder)
import os, hashlib
def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()
have_panel = os.path.exists("panel_spark.parquet") or os.path.exists("panel.parquet")
if not (have_panel and os.path.exists("polars_check.csv")):
    from google.colab import files
    files.upload()
PANEL = "panel_spark.parquet" if os.path.exists("panel_spark.parquet") else "panel.parquet"
print(PANEL, round(os.path.getsize(PANEL) / 1e6, 1), "MB   md5:", md5(PANEL))
print("Compare size and md5 with what 06_export_for_spark.py printed on your computer: they must be identical.")

Saving panel_spark.parquet to panel_spark.parquet
Saving polars_check.csv to polars_check.csv
panel_spark.parquet 67.6 MB   md5: cce9b783d76b2dd9e14d7d757e7093ae
Compare size and md5 with what 06_export_for_spark.py printed on your computer: they must be identical.


In [4]:
import os, time, statistics as st
import pandas as pd
import pyspark
from pyspark.sql import SparkSession, Window, functions as F

RESULTS = []
def say(*a):
    s = " ".join(str(x) for x in a); print(s); RESULTS.append(s)

spark = (SparkSession.builder.master("local[*]").appName("bft65b")
         .config("spark.sql.shuffle.partitions", "8")     # default 200 is far too many for 1.7M rows
         .config("spark.driver.memory", "6g")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
say(f"Spark {pyspark.__version__} local mode, {spark.sparkContext.defaultParallelism} cores, "
    f"machine has {os.cpu_count()} CPU threads")

NEEDED = ["CERT", "qi", "ASSET", "EQ", "NCLNLS", "LNLSGR", "CHBAL", "SC", "LNLSNET", "DEP", "ROA"]

def ensure_spark_readable(path):
    # If Spark's Parquet reader cannot parse the file, rewrite it with pyarrow (another writer) and use that copy.
    try:
        spark.read.parquet(path).select(*NEEDED).limit(1).count()
        return path
    except Exception as e:
        say("Spark cannot read", path, "directly:", str(e).splitlines()[0][:90])
        say("-> rewriting the 11 needed columns with pyarrow into panel_fixed.parquet")
        import pyarrow.parquet as pq
        try:
            tbl = pq.read_table(path, columns=NEEDED)
        except Exception:
            import polars as pl                      # Polars can read files that pyarrow cannot parse
            tbl = pl.read_parquet(path, columns=NEEDED).to_arrow()
        pq.write_table(tbl, "panel_fixed.parquet", compression="snappy")
        return "panel_fixed.parquet"

PANEL = ensure_spark_readable(PANEL)
say("Spark reads:", PANEL)

Spark 4.0.5 local mode, 2 cores, machine has 2 CPU threads
Spark reads: panel_spark.parquet


## 1. The pipeline in Spark (same 11 columns and same definitions as in Polars)

In [5]:
def spark_features(sdf):
    w   = Window.partitionBy("CERT").orderBy("qi")
    w4  = w.rowsBetween(-3, 0)                                   # current + 3 previous quarters (past only)
    wq  = Window.partitionBy("qi").orderBy(F.col("eq_assets").asc_nulls_last())
    wt  = Window.partitionBy("qi", "eq_assets")
    wn  = Window.partitionBy("qi")
    f = (sdf
         .withColumn("eq_assets", F.col("EQ") / F.col("ASSET") * 100)
         .withColumn("npl_ratio", F.when(F.col("LNLSGR") > 0, F.col("NCLNLS") / F.col("LNLSGR") * 100))
         .withColumn("liquid", (F.col("CHBAL") + F.col("SC")) / F.col("ASSET") * 100)
         .withColumn("loans_dep", F.when(F.col("DEP") > 0, F.col("LNLSNET") / F.col("DEP") * 100))
         .withColumn("log_asset", F.log("ASSET"))
         .withColumn("eq_lag4", F.lag("eq_assets", 4).over(w))
         .withColumn("eq_chg4", F.col("eq_assets") - F.col("eq_lag4"))
         .withColumn("roa_m4", F.when(F.count("ROA").over(w4) == 4, F.avg("ROA").over(w4)))
         .withColumn("roa_s4", F.when(F.count("ROA").over(w4) == 4, F.stddev("ROA").over(w4))))
    mx = sdf.groupBy("CERT").agg(F.max("ASSET").alias("max_asset"))
    f = f.join(mx, "CERT", "left")
    # Spark has no 'average' rank: average rank = min rank + (number of ties - 1) / 2
    f = (f.withColumn("rk", F.rank().over(wq))
          .withColumn("ties", F.count(F.lit(1)).over(wt))
          .withColumn("nn", F.count("eq_assets").over(wn))
          .withColumn("eq_rank", F.when(F.col("eq_assets").isNotNull(),
                                        (F.col("rk") + (F.col("ties") - 1) / 2) / F.col("nn"))))
    return f

def quarter_summary(f):
    return (f.groupBy("qi").agg(
        F.count(F.lit(1)).alias("n"), F.avg("eq_assets").alias("mean_eq_assets"),
        F.avg("npl_ratio").alias("mean_npl_ratio"), F.avg("liquid").alias("mean_liquid"),
        F.avg("loans_dep").alias("mean_loans_dep"), F.avg("log_asset").alias("mean_log_asset"),
        F.avg("eq_lag4").alias("mean_eq_lag4"), F.avg("eq_chg4").alias("mean_eq_chg4"),
        F.avg("roa_m4").alias("mean_roa_m4"), F.avg("roa_s4").alias("mean_roa_s4"),
        F.sum("max_asset").alias("sum_max_asset"), F.avg("eq_rank").alias("mean_eq_rank")).orderBy("qi"))

sdf = spark.read.parquet(PANEL).select(*NEEDED)
summary = quarter_summary(spark_features(sdf))
res = summary.toPandas()
say(f"Spark result: {len(res)} quarters, {int(res['n'].sum()):,} rows")

Spark result: 170 quarters, 1,676,980 rows


## 2. Do Spark and Polars give the same numbers?

In [6]:
ref = pd.read_csv("polars_check.csv")
m = res.merge(ref, on="qi", suffixes=("_spark", "_polars"))
say(f"Quarters in Spark: {len(res)}, in Polars file: {len(ref)}, matched on qi: {len(m)}")
rows = []
for c in [c for c in ref.columns if c != "qi"]:
    a, b = m[c + "_spark"].astype(float), m[c + "_polars"].astype(float)
    both_nan = a.isna() & b.isna()
    rel = ((a - b).abs() / (b.abs() + 1e-12))[~both_nan]
    rows.append({"column": c, "max_relative_difference": rel.max(), "nan_mismatch": int((a.isna() ^ b.isna()).sum())})
cmp_df = pd.DataFrame(rows)
say(cmp_df.to_string(index=False))
worst = cmp_df.max_relative_difference.max()
say(f"\nWorst relative difference over all columns and quarters: {worst:.2e} -> "
    + ("RESULTS MATCH (tolerance 1e-9)" if worst < 1e-9 and cmp_df.nan_mismatch.sum() == 0 else "CHECK DIFFERENCES"))

Quarters in Spark: 170, in Polars file: 170, matched on qi: 170
        column  max_relative_difference  nan_mismatch
             n             0.000000e+00             0
mean_eq_assets             7.490783e-15             0
mean_npl_ratio             9.703778e-15             0
   mean_liquid             6.903839e-15             0
mean_loans_dep             9.495927e-15             0
mean_log_asset             7.515630e-15             0
  mean_eq_lag4             6.190104e-15             0
  mean_eq_chg4             3.830501e-14             0
   mean_roa_m4             8.272948e-15             0
   mean_roa_s4             1.033745e-14             0
 sum_max_asset             0.000000e+00             0
  mean_eq_rank             1.820460e-14             0

Worst relative difference over all columns and quarters: 3.83e-14 -> RESULTS MATCH (tolerance 1e-9)


## 3. Execution plans

In [7]:
import io, contextlib, re
from collections import Counter
def plan_text(df):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode="formatted")
    return buf.getvalue()

say("=== Plan A: filter + select + group by (compare with Polars 'Query 2') ===")
qa = sdf.filter(F.col("qi") >= 8000).select("CERT", "qi", "EQ").groupBy("qi").agg(F.sum("EQ").alias("EQ"))
say(plan_text(qa))

=== Plan A: filter + select + group by (compare with Polars 'Query 2') ===
== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Filter (2)
            +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [qi#28, EQ#30]
Batched: true
Location: InMemoryFileIndex [file:/content/panel_spark.parquet]
PushedFilters: [IsNotNull(qi), GreaterThanOrEqual(qi,8000)]
ReadSchema: struct<qi:int,EQ:double>

(2) Filter
Input [2]: [qi#28, EQ#30]
Condition : (isnotnull(qi#28) AND (qi#28 >= 8000))

(3) HashAggregate
Input [2]: [qi#28, EQ#30]
Keys [1]: [qi#28]
Functions [1]: [partial_sum(EQ#30)]
Aggregate Attributes [1]: [sum#541]
Results [2]: [qi#28, sum#542]

(4) Exchange
Input [2]: [qi#28, sum#542]
Arguments: hashpartitioning(qi#28, 8), ENSURE_REQUIREMENTS, [plan_id=718]

(5) HashAggregate
Input [2]: [qi#28, sum#542]
Keys [1]: [qi#28]
Functions [1]: [sum(EQ#30)]
Aggregate Attributes [1]: [sum(EQ#30)#540]
Results [2]: [qi#28, sum(EQ#30)#

In [8]:
say("=== Plan B: whole feature pipeline, per-quarter summary ===")
tb = plan_text(summary)
print(tb)                                   # full plan (long): read it, but you do not need to copy it
nodes = Counter(re.findall(r"^\(\d+\) (\w+)", tb, flags=re.M))
say("Plan B: number of nodes by type:", dict(nodes))
say("Plan B: shuffles (Exchange):", re.findall(r"Arguments: ((?:hash|range)partitioning\([^)]*\)|SinglePartition)", tb))

=== Plan B: whole feature pipeline, per-quarter summary ===
== Physical Plan ==
AdaptiveSparkPlan (61)
+- == Final Plan ==
   ResultQueryStage (37)
   +- * Sort (36)
      +- AQEShuffleRead (35)
         +- ShuffleQueryStage (34), Statistics(sizeInBytes=18.6 KiB, rowCount=170)
            +- Exchange (33)
               +- * HashAggregate (32)
                  +- * HashAggregate (31)
                     +- * Project (30)
                        +- Window (29)
                           +- * Sort (28)
                              +- Window (27)
                                 +- Window (26)
                                    +- * Sort (25)
                                       +- AQEShuffleRead (24)
                                          +- ShuffleQueryStage (23), Statistics(sizeInBytes=153.5 MiB, rowCount=1.68E+6)
                                             +- Exchange (22)
                                                +- * Project (21)
                                     

**How to read the plans** (write your own interpretation in the report):
* `FileScan parquet ... ReadSchema` / `PushedFilters` - Spark also reads only the needed columns and pushes the filter into the scan, like Polars.
* `Exchange hashpartitioning(CERT ...)` / `Exchange hashpartitioning(qi ...)` - a **shuffle**: rows are redistributed across partitions
  so that all rows of one bank (or one quarter) are together. Every window function and join needs one. On a cluster this means network traffic.
* `Sort`, `Window` - window functions per partition; `HashAggregate` with partial + final steps - aggregation done in two stages.
* `AdaptiveSparkPlan` - Spark may re-optimise at run time (e.g. merging small partitions).

## 4. Run time on the same machine: Spark vs Polars

In [9]:
import polars as pl

def pl_ratios(x):
    return x.with_columns(
        (pl.col("EQ") / pl.col("ASSET") * 100).alias("eq_assets"),
        pl.when(pl.col("LNLSGR") > 0).then(pl.col("NCLNLS") / pl.col("LNLSGR") * 100).alias("npl_ratio"),
        ((pl.col("CHBAL") + pl.col("SC")) / pl.col("ASSET") * 100).alias("liquid"),
        pl.when(pl.col("DEP") > 0).then(pl.col("LNLSNET") / pl.col("DEP") * 100).alias("loans_dep"),
        pl.col("ASSET").log().alias("log_asset"))
def pl_window(x):
    return x.with_columns(
        pl.col("eq_assets").shift(4).over("CERT").alias("eq_lag4"),
        pl.col("ROA").rolling_mean(window_size=4, min_samples=4).over("CERT").alias("roa_m4"),
        pl.col("ROA").rolling_std(window_size=4, min_samples=4).over("CERT").alias("roa_s4"),
    ).with_columns((pl.col("eq_assets") - pl.col("eq_lag4")).alias("eq_chg4"))
def pl_join(x):
    return x.join(x.group_by("CERT").agg(pl.col("ASSET").max().alias("max_asset")), on="CERT", how="left")
def pl_rank(x):
    return x.with_columns((pl.col("eq_assets").rank("average").over("qi")
                           / pl.col("eq_assets").count().over("qi")).alias("eq_rank"))
def polars_run(path=None):
    path = path or PANEL
    lf = pl_rank(pl_join(pl_window(pl_ratios(pl.scan_parquet(path).select(NEEDED)))))
    return (lf.group_by("qi").agg(pl.len().alias("n"), pl.col("eq_assets").mean(), pl.col("eq_rank").mean(),
                                  pl.col("roa_s4").mean()).sort("qi").collect())

def spark_run():
    return quarter_summary(spark_features(spark.read.parquet(PANEL).select(*NEEDED))).toPandas()

def timeit(fn, n=3):
    fn()                                   # warm-up
    ts = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return st.mean(ts), st.stdev(ts)

mp, sp = timeit(polars_run, 5)
ms, ss = timeit(spark_run, 3)
say(f"Same machine ({os.cpu_count()} CPU threads), file -> per-quarter summary, 1x data ({int(res['n'].sum()):,} rows):")
say(f"  Polars (lazy) : {mp:6.2f} +- {sp:4.2f} s")
say(f"  Spark (local) : {ms:6.2f} +- {ss:4.2f} s   -> Spark is {ms / mp:.1f}x slower")

Same machine (2 CPU threads), file -> per-quarter summary, 1x data (1,676,980 rows):
  Polars (lazy) :   0.64 +- 0.01 s
  Spark (local) :  22.00 +- 1.67 s   -> Spark is 34.6x slower


## 5. Larger data (replicated panel): does Spark catch up?

In [10]:
from functools import reduce
def spark_k(k):
    base = spark.read.parquet(PANEL).select(*NEEDED)
    return reduce(lambda a, b: a.unionByName(b),
                  [base.withColumn("CERT", F.col("CERT") + i * 1_000_000) for i in range(k)])
def polars_k(k):
    base = pl.scan_parquet(PANEL).select(NEEDED)
    lf = pl.concat([base.with_columns(pl.col("CERT") + i * 1_000_000) for i in range(k)])
    lf = pl_rank(pl_join(pl_window(pl_ratios(lf))))
    return lf.group_by("qi").agg(pl.len().alias("n"), pl.col("eq_assets").mean()).sort("qi").collect()

say("rows (millions) | Polars s | Spark s")
for k in [1, 4]:
    rows_m = int(res["n"].sum()) * k / 1e6
    a, _ = timeit(lambda: polars_k(k), 2)
    b, _ = timeit(lambda: quarter_summary(spark_features(spark_k(k))).toPandas(), 2)
    say(f"{rows_m:14.1f} | {a:8.2f} | {b:7.2f}")

rows (millions) | Polars s | Spark s
           1.7 |     0.27 |   21.79
           6.7 |     0.63 |   79.12


## 6. Copy the block below into the chat

In [11]:
print("\n".join(RESULTS))

Spark 4.0.5 local mode, 2 cores, machine has 2 CPU threads
Spark reads: panel_spark.parquet
Spark result: 170 quarters, 1,676,980 rows
Quarters in Spark: 170, in Polars file: 170, matched on qi: 170
        column  max_relative_difference  nan_mismatch
             n             0.000000e+00             0
mean_eq_assets             7.490783e-15             0
mean_npl_ratio             9.703778e-15             0
   mean_liquid             6.903839e-15             0
mean_loans_dep             9.495927e-15             0
mean_log_asset             7.515630e-15             0
  mean_eq_lag4             6.190104e-15             0
  mean_eq_chg4             3.830501e-14             0
   mean_roa_m4             8.272948e-15             0
   mean_roa_s4             1.033745e-14             0
 sum_max_asset             0.000000e+00             0
  mean_eq_rank             1.820460e-14             0

Worst relative difference over all columns and quarters: 3.83e-14 -> RESULTS MATCH (tolerance 1e-9